# NT Pipeline: Uniform Chapter JSON + Validation

**First pass** – standardise data storage so every chapter folder has one dense-whitespace JSON (like OT branch).

1. **Scan** each subfolder of `bible/` – find numbered `.json` and `.html` files  
2. **JSON cleanup** – remove `"index"` keys, keep dense formatting  
3. **HTML-only chapters** (html exists, json missing) – extract mapping data into new JSON  
4. **Validate** greek words against strongs CSV, latvian words against lv65 CSV

## 0 – Config & Imports

In [47]:
from pathlib import Path
import json, re, unicodedata, os
from collections import Counter, defaultdict
from bs4 import BeautifulSoup
import pandas as pd

BASE_DIR = Path("bible")          # <── set your root here
DRY_RUN  = True                    # True = preview only, False = write files
VERBOSE  = True

## 1 – Load reference datasets

In [48]:
strongs_df = pd.read_csv("strongs.csv")
lv65_df    = pd.read_csv("latvian_full65.csv")

strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
lv_g      = lv65_df.groupby(["book", "chapter", "verse"])

print(f"Strongs: {len(strongs_df)} rows, LV65: {len(lv65_df)} rows")
print("Strongs columns:", list(strongs_df.columns))
print("LV65 columns:",    list(lv65_df.columns))

Strongs: 138993 rows, LV65: 7956 rows
Strongs columns: ['verse', 'word', 'form', 'form_en', 'strong_num', 'strong_title', 'strong_en_title', 'translit', 'translit_title', 'book', 'chapter']
LV65 columns: ['book', 'chapter', 'verse', 'text']


## 2 – Scan folders: discover JSON & HTML files

In [49]:
def discover_numbered_files(folder: Path):
    """Return dict of {int: extension} for files like 1.json, 2.html, etc."""
    jsons = set()
    htmls = set()
    for f in folder.iterdir():
        if f.is_file() and f.stem.isdigit():
            n = int(f.stem)
            if f.suffix == '.json':
                jsons.add(n)
            elif f.suffix == '.html':
                htmls.add(n)
    return sorted(jsons), sorted(htmls)

# Scan all book subfolders
book_scan = {}
for book_dir in sorted(BASE_DIR.iterdir()):
    if not book_dir.is_dir():
        continue
    jsons, htmls = discover_numbered_files(book_dir)
    if jsons or htmls:
        html_only = sorted(set(htmls) - set(jsons))
        json_only = sorted(set(jsons) - set(htmls))
        both      = sorted(set(jsons) & set(htmls))
        book_scan[book_dir.name] = {
            'jsons': jsons, 'htmls': htmls,
            'both': both, 'html_only': html_only, 'json_only': json_only
        }
        print(f"📁 {book_dir.name:>18}: {len(jsons):^3} json, {len(htmls):^3} html | "
              f"both={len(both):^3} html_only={len(html_only):^3} json_only={len(json_only):^3}")

print(f"\nTotal books: {len(book_scan)}")

📁      1_corinthians: 16  json, 16  html | both=16  html_only= 0  json_only= 0 
📁             1_john:  5  json,  5  html | both= 5  html_only= 0  json_only= 0 
📁            1_peter:  5  json,  5  html | both= 5  html_only= 0  json_only= 0 
📁    1_thessalonians:  5  json,  5  html | both= 5  html_only= 0  json_only= 0 
📁          1_timothy:  6  json,  6  html | both= 6  html_only= 0  json_only= 0 
📁      2_corinthians: 13  json, 13  html | both=13  html_only= 0  json_only= 0 
📁             2_john:  1  json,  1  html | both= 1  html_only= 0  json_only= 0 
📁            2_peter:  3  json,  3  html | both= 3  html_only= 0  json_only= 0 
📁    2_thessalonians:  3  json,  3  html | both= 3  html_only= 0  json_only= 0 
📁          2_timothy:  4  json,  4  html | both= 4  html_only= 0  json_only= 0 
📁             3_john:  1  json,  1  html | both= 1  html_only= 0  json_only= 0 
📁               acts: 28  json, 28  html | both=28  html_only= 0  json_only= 0 
📁         colossians:  4  json,  4  html

## 3 – JSON cleanup: remove `index` keys

For each existing `N.json`, strip `"index": N, ` from every `greek_words` entry.  
Uses regex on the raw text to preserve the dense single-line-per-word formatting.

In [50]:
INDEX_RE = re.compile(r'"index":\s*\d+,\s*')
#DRY_RUN=False
def strip_index_from_json_file(json_path: Path, dry_run=True):
    """
    Remove all '"index": N, ' from the raw JSON text.
    Returns (changed: bool, count_removed: int).
    """
    raw = json_path.read_text(encoding='utf-8')
    new_raw, count = INDEX_RE.subn('', raw)
    
    if count == 0:
        return False, 0
    
    # Validate the result is still valid JSON
    try:
        json.loads(new_raw)
    except json.JSONDecodeError as e:
        print(f"  ❌ JSON broken after index removal in {json_path}: {e}")
        return False, 0
    
    if not dry_run:
        json_path.write_text(new_raw, encoding='utf-8')
    
    return True, count

# Run on all discovered JSONs
total_files_changed = 0
total_indexes_removed = 0

for book_name, info in book_scan.items():
    book_dir = BASE_DIR / book_name
    for ch_num in info['jsons']:
        json_path = book_dir / f"{ch_num}.json"
        changed, count = strip_index_from_json_file(json_path, dry_run=DRY_RUN)
        if changed:
            total_files_changed += 1
            total_indexes_removed += count
            if VERBOSE:
                print(f"  {'🔍' if DRY_RUN else '✅'} {json_path}: removed {count} index entries")

action = "Would change" if DRY_RUN else "Changed"
print(f"\n{action} {total_files_changed} file(s), {total_indexes_removed} index entries total.")
if DRY_RUN:
    print("Set DRY_RUN = False to write changes.")


Would change 0 file(s), 0 index entries total.
Set DRY_RUN = False to write changes.


## 4 – Extract JSON from HTML-only chapters

For chapters that have an `.html` but no `.json`, parse the HTML and produce
a dense-format JSON matching the OT standard:
```json
[
  {
    "greek_words": [
      { "greek": "...", "latvian": [...] },
      ...
    ],
    "leftover_latvian": [...]
  },
  ...
]
```

In [51]:
def extract_verses_from_html(html_path: Path):
    """
    Parse an NT chapter HTML and extract verse data.
    Returns a list of verse dicts with greek_words and leftover_latvian.
    """
    raw = html_path.read_text(encoding='utf-8')
    soup = BeautifulSoup(raw, 'html.parser')
    
    verses = []
    
    for container in soup.find_all('div', class_='verse-container'):
        greek_words = []
        leftover_latvian = []
        
        table = container.find('table', class_='mapping-table')
        if not table:
            verses.append({'greek_words': [], 'leftover_latvian': []})
            continue
        
        tbody = table.find('tbody')
        if not tbody:
            verses.append({'greek_words': [], 'leftover_latvian': []})
            continue
        
        for row in tbody.find_all('tr'):
            tds = row.find_all('td')
            if not tds:
                continue
            
            # Check if this is a "no match" / leftover row
            no_match_span = tds[0].find('span', class_='greek-form')
            if no_match_span and '(no match)' in no_match_span.get_text():
                # leftover row: td[1] has colspan=4 with comma-separated words
                if len(tds) >= 2:
                    leftover_text = tds[1].get_text(strip=True)
                    # Split by ' ,' pattern (the format from the renderer)
                    if leftover_text and leftover_text != '-':
                        parts = [w.strip() for w in leftover_text.split(',')]
                        leftover_latvian.extend(p for p in parts if p)
                continue
            
            # Normal word row: td[0]=greek, td[1]=latvian, td[2]=strongs, ...
            if len(tds) < 2:
                continue
            
            # Extract Greek word from td[0]
            greek_td = tds[0]
            # The greek word is the direct text of the td (before the span)
            # Structure: <td class="greek-word">Ἤκουσαν <span class="greek-form">(Ēkousan)</span></td>
            # Must use isinstance(Tag) because NavigableString also has .name=None
            from bs4 import Tag as _Tag
            greek_text = ''
            for child in greek_td.children:
                if isinstance(child, _Tag):
                    if child.name in ('span', 'br', 'audio', 'button'):
                        break
                else:
                    greek_text += str(child)
            greek_text = greek_text.strip()
            # Strip punctuation artifacts from HTML display: «» [] () ' ⇔ etc.
            greek_text = re.sub(r'[«»\[\]()⇔·;,.!?᾽᾿ʼ]', '', greek_text)
            # Also strip smart/curly apostrophes that look like Greek breathings
            greek_text = greek_text.replace("\u2019", "").replace("\u2018", "")
            greek_text = greek_text.replace("'", "").replace("'", "")
            greek_text = greek_text.strip()
            
            greek_stripped = strip_greek_diacritics(greek_text)
            
            # Extract Latvian words from td[1]
            latvian_td = tds[1]
            latvian_text = latvian_td.get_text(strip=True)
            
            if latvian_text == '-' or not latvian_text:
                latvian_list = []
            else:
                # Words are comma+space separated: "esmu, stāstījis"
                latvian_list = [w.strip() for w in latvian_text.split(',') if w.strip()]
            
            greek_words.append({
                'greek': greek_stripped,
                'latvian': latvian_list
            })
        
        verses.append({
            'greek_words': greek_words,
            'leftover_latvian': leftover_latvian
        })
    
    return verses


def strip_greek_diacritics(s):
    """Remove Greek diacritics/accents, keep base letters."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

In [52]:
def verses_to_dense_json(verses):
    """
    Serialize verse list to the dense compact JSON format:
    one greek_word per line, verse-level keys on own lines.
    Matches the OT chapter JSON style.
    """
    def word_to_compact(w):
        gr  = json.dumps(w.get('greek', ''), ensure_ascii=False)
        lv  = json.dumps(w.get('latvian', []), ensure_ascii=False)
        return f'{{ "greek": {gr}, "latvian": {lv} }}'
    
    verse_blocks = []
    for verse in verses:
        words = verse.get('greek_words', [])
        compact_words = ',\n      '.join(word_to_compact(w) for w in words)
        lo_lv = json.dumps(verse.get('leftover_latvian', []), ensure_ascii=False)
        verse_blocks.append(
            f'  {{\n    "greek_words": [\n      {compact_words}\n    ],\n'
            f'    "leftover_latvian": {lo_lv}\n  }}'
        )
    
    return '[\n' + ',\n'.join(verse_blocks) + '\n]'

In [53]:
# Extract JSON from all html-only chapters
extracted_count = 0

for book_name, info in book_scan.items():
    book_dir = BASE_DIR / book_name
    for ch_num in info['html_only']:
        html_path = book_dir / f"{ch_num}.html"
        json_path = book_dir / f"{ch_num}.json"
        
        verses = extract_verses_from_html(html_path)
        
        if not verses:
            print(f"  ⚠️ {html_path}: no verses extracted!")
            continue
        
        dense_json = verses_to_dense_json(verses)
        
        # Validate
        try:
            parsed = json.loads(dense_json)
            assert len(parsed) == len(verses)
        except (json.JSONDecodeError, AssertionError) as e:
            print(f"  ❌ {html_path}: generated JSON is invalid: {e}")
            continue
        
        total_words = sum(len(v['greek_words']) for v in verses)
        total_leftovers = sum(len(v['leftover_latvian']) for v in verses)
        
        if DRY_RUN:
            print(f"  🔍 {html_path} → would create {json_path}: "
                  f"{len(verses)} verses, {total_words} words, {total_leftovers} leftovers")
        else:
            json_path.write_text(dense_json, encoding='utf-8')
            print(f"  ✅ {json_path}: {len(verses)} verses, {total_words} words, {total_leftovers} leftovers")
        
        extracted_count += 1

action = "Would extract" if DRY_RUN else "Extracted"
print(f"\n{action} {extracted_count} chapter(s) from HTML.")


Would extract 0 chapter(s) from HTML.


### 4b – Spot-check: compare extracted JSON vs existing JSON for chapter 11

Since chapter 11 has both HTML and JSON, we can extract from HTML and compare with the existing JSON to verify the extractor works correctly.

In [54]:
def spot_check_extraction(book_name, ch_num):
    """
    For a chapter that has BOTH json and html, extract from HTML
    and compare verse-by-verse against the existing JSON.
    """
    book_dir  = BASE_DIR / book_name
    json_path = book_dir / f"{ch_num}.json"
    html_path = book_dir / f"{ch_num}.html"
    
    if not json_path.exists() or not html_path.exists():
        print(f"Need both {json_path} and {html_path} for spot-check")
        return
    
    # Load existing JSON (with or without index)
    with open(json_path, 'r', encoding='utf-8') as f:
        existing = json.load(f)
    
    # Extract from HTML
    extracted = extract_verses_from_html(html_path)
    
    print(f"Existing JSON: {len(existing)} verses")
    print(f"Extracted from HTML: {len(extracted)} verses")
    
    if len(existing) != len(extracted):
        print(f"❌ Verse count mismatch!")
        return
    
    mismatches = 0
    for vi in range(len(existing)):
        ex_words = existing[vi].get('greek_words', [])
        ht_words = extracted[vi].get('greek_words', [])
        
        if len(ex_words) != len(ht_words):
            print(f"  ⚠️ v{vi+1}: word count {len(ex_words)} vs {len(ht_words)}")
            mismatches += 1
            continue
        
        for wi in range(len(ex_words)):
            # Compare greek (strip accents for comparison since JSON has stripped, HTML has accented)
            ex_gr = ex_words[wi].get('greek', '')
            ht_gr = ht_words[wi].get('greek', '')
            
            # Compare stripped forms
            ex_stripped = strip_greek_diacritics(ex_gr)
            ht_stripped = strip_greek_diacritics(ht_gr)
            
            if ex_stripped.lower() != ht_stripped.lower():
                print(f"  ⚠️ v{vi+1} w{wi+1}: greek '{ex_gr}' vs '{ht_gr}'")
                mismatches += 1
            
            # Compare latvian
            ex_lv = ex_words[wi].get('latvian', [])
            ht_lv = ht_words[wi].get('latvian', [])
            if ex_lv != ht_lv:
                print(f"  ⚠️ v{vi+1} w{wi+1}: latvian {ex_lv} vs {ht_lv}")
                mismatches += 1
        
        # Compare leftovers
        ex_lo = existing[vi].get('leftover_latvian', [])
        ht_lo = extracted[vi].get('leftover_latvian', [])
        if ex_lo != ht_lo:
            print(f"  ⚠️ v{vi+1}: leftover_latvian {ex_lo} vs {ht_lo}")
            mismatches += 1
    
    if mismatches == 0:
        print("✅ Perfect match between existing JSON and HTML extraction!")
    else:
        print(f"⚠️ {mismatches} mismatch(es) found – review above.")

# Run spot-check on chapters that have both html and json
for book_name, info in book_scan.items():
    for ch_num in info['both'][:2]:  # check first 2 per book
        print(f"\n--- Spot-check: {book_name}/{ch_num} ---")
        spot_check_extraction(book_name, ch_num)


--- Spot-check: 1_corinthians/1 ---
Existing JSON: 31 verses
Extracted from HTML: 31 verses
✅ Perfect match between existing JSON and HTML extraction!

--- Spot-check: 1_corinthians/2 ---
Existing JSON: 16 verses
Extracted from HTML: 16 verses
✅ Perfect match between existing JSON and HTML extraction!

--- Spot-check: 1_john/1 ---
Existing JSON: 10 verses
Extracted from HTML: 10 verses
✅ Perfect match between existing JSON and HTML extraction!

--- Spot-check: 1_john/2 ---
Existing JSON: 29 verses
Extracted from HTML: 29 verses
✅ Perfect match between existing JSON and HTML extraction!

--- Spot-check: 1_peter/1 ---
Existing JSON: 25 verses
Extracted from HTML: 25 verses
✅ Perfect match between existing JSON and HTML extraction!

--- Spot-check: 1_peter/2 ---
Existing JSON: 25 verses
Extracted from HTML: 25 verses
✅ Perfect match between existing JSON and HTML extraction!

--- Spot-check: 1_thessalonians/1 ---
Existing JSON: 10 verses
Extracted from HTML: 10 verses
✅ Perfect match bet

In [55]:
strongs_df.query(
    "book=='romans' & chapter==2 & verse==16 & word==1"
)

,verse,word,form,form_en,strong_num,strong_title,strong_en_title,translit,translit_title,book,chapter
134841,16,1,ᾗ,[that],3739,"3739: Who, which, what, that.",RelPro-DFS,hē,"hē: Who, which, what, that.",romans,2


## 5 – Validation helpers

In [56]:
# ── Latvian helpers (from OT pipeline) ──────────────────────────────────────

LV_CHARS = "A-Za-zāēīūģķļņšžčĀĒĪŪĢĶĻŅŠŽČ"
LV_REGEX = re.compile(f"[{LV_CHARS}]+")

def lv_tokens(text: str):
    """Return list of Latvian word-tokens from any string."""
    return LV_REGEX.findall(unicodedata.normalize('NFC', text))

def strip_lv_diacritics(s: str) -> str:
    """Lower-case and strip Latvian diacritics for fuzzy comparison."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s.lower())
        if unicodedata.category(c) != 'Mn'
    )


# ── Greek helpers ────────────────────────────────────────────────────────────

def strip_greek_diacritics(s):
    """Remove Greek diacritics/accents, keep base letters."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def is_greek_diacritic_diff(ref, json_w):
    """True if Greek words differ only by accents/diacritics.
    Skip single-char words where diacritic changes the word entirely."""
    stripped_ref  = strip_greek_diacritics(ref)
    stripped_json = strip_greek_diacritics(json_w)
    if len(stripped_ref) <= 1:
        return False
    return stripped_ref == stripped_json

In [57]:
# ── ANSI colours for diff ────────────────────────────────────────────────────

GREEN    = '\033[92m'
RED_W    = '\x1b[38;5;88m\x1b[48;5;7m'
DIFF_BLUE = '\x1b[38;5;12m\x1b[48;15;7m'
DIFF_RED  = '\x1b[38;5;196m\x1b[48;15;7m'
RESET    = '\033[0m'

## 6 – Validate: Strongs (Greek) word check

For each verse in a chapter JSON, compare the greek words list  
against the strongs CSV reference (same logic as OT but simpler –  
greek is the base language, no hebrew layer).

In [58]:
def validate_strongs_for_chapter(book_name, chapter_num, strongs_g, verbose=True):
    """
    Compare greek_words in the chapter JSON against strongs CSV.
    Reports:
      - word count mismatches
      - word-level diffs (diacritics-only vs real diff)
    Returns list of issue dicts.
    """
    json_path = BASE_DIR / book_name / f"{chapter_num}.json"
    if not json_path.exists():
        return [{'type': 'missing_json', 'book': book_name, 'chapter': chapter_num}]
    
    with open(json_path, 'r', encoding='utf-8') as f:
        verses = json.load(f)
    
    issues = []
    
    for vi, verse_data in enumerate(verses):
        verse_num = vi + 1
        key = (book_name, chapter_num, verse_num)
        
        if key not in strongs_g.groups:
            if verbose:
                print(f"  ⚠️🇬🇷 No strongs ref for {key}")
            issues.append({'type': 'missing_strongs', 'key': key})
            continue
        
        ref_strongs = strongs_g.get_group(key).sort_values('word')
        json_greek = verse_data.get('greek_words', [])
        
        ref_count  = len(ref_strongs)
        json_count = len(json_greek)
        
        if ref_count != json_count:
            if verbose:
                print(f"  ❌🇬🇷 v{verse_num} word count: strongs={ref_count} json={json_count}")
            issues.append({
                'type': 'count_mismatch', 'key': key,
                'ref': ref_count, 'json': json_count
            })
        
        # Word-by-word comparison
        min_len = min(ref_count, json_count)
        for wi in range(min_len):
            ref_form   = str(ref_strongs.iloc[wi]['form'])
            json_greek_word = json_greek[wi].get('greek', '')
            
            ref_stripped  = strip_greek_diacritics(ref_form)
            json_stripped = strip_greek_diacritics(json_greek_word)
            
            if ref_stripped.lower() != json_stripped.lower():
                if verbose:
                    print(f"  Δ🇬🇷 v{verse_num} w{wi+1}: ref='{ref_form}' json='{json_greek_word}'")
                issues.append({
                    'type': 'greek_diff', 'key': key, 'word_idx': wi+1,
                    'ref': ref_form, 'json': json_greek_word
                })
    
    return issues

## 7 – Validate: Latvian word check

For each verse, check that all words from the lv65 reference text  
appear somewhere in the mapped latvian arrays + leftover_latvian.  
Same logic as OT `validate_latvian_usage` with the `ne-` prefix exception.

In [59]:
def validate_latvian_for_chapter(book_name, chapter_num, lv_g, verbose=True):
    """
    For each verse, check that all Latvian ref tokens are accounted for
    in greek_words[*].latvian + leftover_latvian.
    Uses diacritics-stripped comparison with the 'ne-' prefix exception.
    Returns list of issue dicts.
    """
    json_path = BASE_DIR / book_name / f"{chapter_num}.json"
    if not json_path.exists():
        return [{'type': 'missing_json', 'book': book_name, 'chapter': chapter_num}]
    
    with open(json_path, 'r', encoding='utf-8') as f:
        verses = json.load(f)
    
    issues = []
    
    for vi, verse_data in enumerate(verses):
        verse_num = vi + 1
        key = (book_name, chapter_num, verse_num)
        
        if key not in lv_g.groups:
            if verbose:
                print(f"  ⚠️🇱🇻 No latvian ref for {key}")
            issues.append({'type': 'missing_lv', 'key': key})
            continue
        
        # Get reference latvian text
        lv_text = lv_g.get_group(key).iloc[0]['text']
        ref_tokens = lv_tokens(lv_text)
        
        if not ref_tokens:
            continue
        
        # Collect all mapped latvian words from JSON
        mapped_words = []
        for gw in verse_data.get('greek_words', []):
            for item in gw.get('latvian', []):
                if item and item != '-':
                    clean = unicodedata.normalize('NFC', item)
                    mapped_words.extend(lv_tokens(clean))
        
        for item in verse_data.get('leftover_latvian', []):
            clean = unicodedata.normalize('NFC', item)
            mapped_words.extend(lv_tokens(clean))
        
        # Compare with diacritics-stripped matching
        ref_counts    = Counter(ref_tokens)
        mapped_counts = Counter(mapped_words)
        
        missing = []
        for word, count in ref_counts.items():
            stripped_word = strip_lv_diacritics(word)
            matched = sum(
                v for k, v in mapped_counts.items()
                if strip_lv_diacritics(k) == stripped_word
            )
            missing_count = count - matched
            if missing_count > 0:
                # 'ne-' prefix exception
                lw = word.lower()
                if len(lw) > 2 and lw.startswith('ne'):
                    stem = word[2:]
                    stem_stripped = strip_lv_diacritics(stem)
                    stem_matched = sum(
                        v for k, v in mapped_counts.items()
                        if strip_lv_diacritics(k) == stem_stripped
                    )
                    if stem_matched >= count:
                        continue
                
                for _ in range(missing_count):
                    missing.append(word)
        
        if missing:
            if verbose:
                print(f"  ⚠️🇱🇻 v{verse_num} missing: {', '.join(missing)}")
            issues.append({
                'type': 'latvian_missing', 'key': key,
                'missing': missing
            })
    
    return issues

## 8 – Run validation on all chapters

In [60]:
all_strongs_issues = []
all_latvian_issues = []

for book_name, info in book_scan.items():
    all_chapters = sorted(set(info['jsons']) | set(info['html_only']))
    # After step 3+4, all chapters should have JSONs
    # (html_only ones got new JSONs if DRY_RUN was False)
    
    for ch_num in all_chapters:
        json_path = BASE_DIR / book_name / f"{ch_num}.json"
        if not json_path.exists():
            # Still in DRY_RUN for html_only chapters, skip
            continue
        
        print(f"\n{'='*50}")
        print(f"Validating {book_name} chapter {ch_num}")
        print(f"{'='*50}")
        
        si = validate_strongs_for_chapter(book_name, ch_num, strongs_g, verbose=VERBOSE)
        li = validate_latvian_for_chapter(book_name, ch_num, lv_g, verbose=VERBOSE)
        
        if not si and not li:
            print("  ✅ All checks passed")
        
        all_strongs_issues.extend(si)
        all_latvian_issues.extend(li)


Validating 1_corinthians chapter 1
  Δ🇬🇷 v2 w8: ref='Κορίνθῳ ' json='Κορινθω'
  Δ🇬🇷 v4 w4: ref='μου' json='‹μου›'

Validating 1_corinthians chapter 2
  Δ🇬🇷 v15 w5: ref='τὰ' json='τα*'

Validating 1_corinthians chapter 3
  Δ🇬🇷 v12 w8: ref='χρυσόν' json='χρυσον*'
  Δ🇬🇷 v16 w13: ref='ὑμῖν ' json='υμιν'

Validating 1_corinthians chapter 4
  Δ🇬🇷 v8 w17: ref='συμβασιλεύσωμεν' json='συμβασιλευσωμεν*'
  Δ🇬🇷 v17 w3: ref='αὐτὸ' json='〈αυτο〉'

Validating 1_corinthians chapter 5
  Δ🇬🇷 v4 w6: ref='ἡμῶν' json='‹ημων›'

Validating 1_corinthians chapter 6
  Δ🇬🇷 v20 w12: ref='καὶ' json='⧼και'
  Δ🇬🇷 v20 w20: ref='θεοῦ' json='θεου⧽'

Validating 1_corinthians chapter 7
  Δ🇬🇷 v9 w9: ref='γαμῆσαι' json='γαμησαι*'
  Δ🇬🇷 v13 w3: ref='εἴ' json='‹ει'
  Δ🇬🇷 v13 w4: ref='τις' json='τις›'
  Δ🇬🇷 v17 w5: ref='ἐμέρισεν' json='εμερισεν*'
  Δ🇬🇷 v38 w6: ref='ἑαυτοῦ ' json='εαυτου'

Validating 1_corinthians chapter 8
  Δ🇬🇷 v8 w15: ref='μὴ' json='‹μη›'
  ⚠️🇱🇻 v8 missing: netuvinās

Validating 1_corinthians chapter 9
  ✅ 

In [61]:
strongs_df.query(
    "book=='revelation' & chapter==14 & verse==5 & word==10"
)


,verse,word,form,form_en,strong_num,strong_title,strong_en_title,translit,translit_title,book,chapter
123563,5,10,ἐνώπιον,before,1799,"1799: Before the face of, in the presence of, ...",Prep,enōpion,"enōpion: Before the face of, in the presence o...",revelation,14


In [62]:
# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("VALIDATION SUMMARY")
print("="*60)

strongs_types = Counter(i['type'] for i in all_strongs_issues)
latvian_types = Counter(i['type'] for i in all_latvian_issues)

print(f"\n🇬🇷 Strongs issues: {len(all_strongs_issues)}")
for t, c in strongs_types.most_common():
    print(f"   {t}: {c}")

print(f"\n🇱🇻 Latvian issues: {len(all_latvian_issues)}")
for t, c in latvian_types.most_common():
    print(f"   {t}: {c}")

if not all_strongs_issues and not all_latvian_issues:
    print("\n🎉 Everything looks clean!")


VALIDATION SUMMARY

🇬🇷 Strongs issues: 728
   greek_diff: 726
   count_mismatch: 2

🇱🇻 Latvian issues: 829
   latvian_missing: 828
   missing_lv: 1


## 9 – (Optional) Rewrite all JSONs in uniform dense format

After index removal and extraction, ensure every JSON uses the exact same  
compact formatting. This re-serializes each file through `verses_to_dense_json`.

In [65]:
REFORMAT_DRY_RUN = False#True  # Set False to actually rewrite

reformatted = 0
for book_name, info in book_scan.items():
    book_dir = BASE_DIR / book_name
    for ch_num in info['jsons']:
        json_path = book_dir / f"{ch_num}.json"
        
        with open(json_path, 'r', encoding='utf-8') as f:
            old_raw = f.read()
            verses  = json.loads(old_raw)
        
        new_raw = verses_to_dense_json(verses)
        
        if old_raw.strip() == new_raw.strip():
            continue  # already in target format
        
        # Validate round-trip
        reparsed = json.loads(new_raw)
        if len(reparsed) != len(verses):
            print(f"  ❌ {json_path}: round-trip verse count mismatch!")
            continue
        
        if not REFORMAT_DRY_RUN:
            json_path.write_text(new_raw, encoding='utf-8')
        
        reformatted += 1
        if VERBOSE:
            print(f"  {'🔍' if REFORMAT_DRY_RUN else '✅'} {json_path}: reformatted")

action = "Would reformat" if REFORMAT_DRY_RUN else "Reformatted"
print(f"\n{action} {reformatted} file(s).")


Reformatted 0 file(s).
